In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality



# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers and sample data
data = data.replace("?", np.nan)

# Complete-case only — no mean / mode imputation
_before = len(data)
_n_missing_rows = int(data.isna().any(axis=1).sum())
data = data.dropna().reset_index(drop=True)
print(
    f"Dropped {_before - len(data)} rows with missing feature values "
    f"(complete-case; no imputation; rows_with_na={_n_missing_rows})"
)
assert not data.isna().any().any(), "Unexpected NaNs remain after dropna"

# Sample after complete-case so NA handling is not confounded by imputation
n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)
print(f"Adult subsample after complete-case: {len(data)} rows")

numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare features, target, and metadata
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

# Set experiment constants and seeds
N_SAMPLES = 10000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Initialize result containers
scores = {}
synthetic_datasets = {}
quality_results = []


Dropped 3620 rows with missing feature values (complete-case; no imputation; rows_with_na=3620)
Adult subsample after complete-case: 1000 rows


In [4]:
# Single run

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Train/test split without leakage

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[
        col for col in train_real.columns
        if col != target_col and col in label_encoders
    ],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)



================ SINGLE RUN ================
Training TabDDPM...
[0]
85
{'num_classes': 4, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(85)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.2981 Sum: 0.2981
Step 1000/1000 MLoss: 0.0 GLoss: 0.2543 Sum: 0.2543
mlp
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Discrete cols: [2, 4]
Num shape:  (10000, 6)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 31.26it/s]|
Column Shapes Score: 24.56%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 283.85it/s]|
Column Pair Trends Score: 0.0%

Overall Score (Average): 12.28%

TabDDPM: 0.1228


In [5]:
# ForestDiffusion

try:
    import traceback

    print("Training ForestDiffusion...")
    synthetic_forestdiffusion = train_forestdiffusion(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_forestdiffusion,
        metadata=train_metadata,
    )

    scores["ForestDiffusion"] = quality.get_score()

    print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))

    del synthetic_forestdiffusion

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("ForestDiffusion Failed:")
    traceback.print_exc()


Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:04<00:00,  3.59it/s]|
Column Shapes Score: 57.51%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 326.19it/s]|
Column Pair Trends Score: 43.88%

Overall Score (Average): 50.7%

ForestDiffusion: 0.507


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': LinearSVC(max_iter=2000, dual='auto', random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            # Extract X and y from the full training DataFrame for the current seed
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            # Extract X and y from the full testing DataFrame for the current seed
            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            # Determine if stratification is possible for y_train_full
            stratify_y_train_full = y_train_full if y_train_full.value_counts().min() >= 2 else None
            if stratify_y_train_full is None:
                print(
                    f"Warning: Cannot stratify training data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `train_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the first train-test split (from train_df to get actual training set for classifier)
            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_train_full
            )

            # Determine if stratification is possible for y_test_full
            # This is usually for real data and should be fine, but included for robustness.
            stratify_y_test_full = y_test_full if y_test_full.value_counts().min() >= 2 else None
            if stratify_y_test_full is None:
                print(
                    f"Warning: Cannot stratify testing data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `test_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the second train-test split (from test_df to get actual testing set for classifier)
            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_test_full
            )

            scaler = StandardScaler().fit(X_train_split)

            X_train_s = scaler.transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train_split)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test_split, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1)

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1)

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1)

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1)

        results.append({
            "Model": name,

            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,

            "Accuracy \u00b1 SD": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 \u00b1 SD": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision \u00b1 SD": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall \u00b1 SD": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}",

            "Accuracy (Mean\u00b1Std)": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 (Mean\u00b1Std)": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision (Mean\u00b1Std)": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall (Mean\u00b1Std)": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )


In [8]:
# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers
data = data.replace("?", np.nan)

# Clean income target into two classes only
data[target_col] = (
    data[target_col]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.strip()
)

print(data[target_col].value_counts())

# Complete-case only — no mean / mode imputation
_before = len(data)
_n_missing_rows = int(data.isna().any(axis=1).sum())
data = data.dropna().reset_index(drop=True)
print(
    f"Dropped {_before - len(data)} rows with missing feature values "
    f"(complete-case; no imputation; rows_with_na={_n_missing_rows})"
)
assert not data.isna().any().any(), "Unexpected NaNs remain after dropna"

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)
print(f"Adult subsample after complete-case: {len(data)} rows")

numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare processed dataset
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

print("\nEncoded income classes:")
print(processed_data[target_col].value_counts())
print(label_encoders[target_col].classes_)


income
<=50K    37155
>50K     11687
Name: count, dtype: int64
Dropped 3620 rows with missing feature values (complete-case; no imputation; rows_with_na=3620)
Adult subsample after complete-case: 1000 rows

Encoded income classes:
income
0    728
1    272
Name: count, dtype: int64
['<=50K' '>50K']


In [9]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = target_col   # Adult dataset target column: income

model_order = ["TabDDPM", "ForestDiffusion"]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy \u00b1 SD",
            "F1 \u00b1 SD",
            "Precision \u00b1 SD",
            "Recall \u00b1 SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    # Align synthetic columns with processed_data integer encoding.
    for col, le in label_encoders.items():
        if col not in synthetic_train_df.columns:
            continue
        vals = synthetic_train_df[col]
        if pd.api.types.is_integer_dtype(vals):
            continue
        str_vals = vals.astype(str).str.strip()
        known = set(le.classes_)
        if set(str_vals.unique()).issubset(known):
            synthetic_train_df[col] = le.transform(str_vals)
        else:
            synthetic_train_df[col] = (
                pd.to_numeric(vals, errors="coerce")
                .fillna(0)
                .round()
                .astype(int)
                .clip(0, len(le.classes_) - 1)
            )

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_real,
        label_col=target_col,
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy \u00b1 SD",
                "F1 \u00b1 SD",
                "Precision \u00b1 SD",
                "Recall \u00b1 SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=('_TRTR', '_TSTR')
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy \u00b1 SD_TRTR",
                "Accuracy \u00b1 SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
7,GradientBoost,0.5250 ± 0.0527,0.4934 ± 0.0538,0.4830 ± 0.0611,0.5250 ± 0.0527
1,SVM-RBF,0.5150 ± 0.0459,0.4011 ± 0.0512,0.3448 ± 0.0676,0.5150 ± 0.0459
8,AdaBoost,0.5150 ± 0.0489,0.4459 ± 0.0490,0.4205 ± 0.0744,0.5150 ± 0.0489
0,LogReg,0.5050 ± 0.0483,0.3947 ± 0.0506,0.3328 ± 0.0535,0.5050 ± 0.0483
5,RandomForest,0.4875 ± 0.0637,0.4637 ± 0.0680,0.4624 ± 0.0761,0.4875 ± 0.0637
3,NaiveBayes,0.4850 ± 0.0580,0.3885 ± 0.0493,0.3749 ± 0.1033,0.4850 ± 0.0580
6,ExtraTrees,0.4775 ± 0.0650,0.4691 ± 0.0680,0.4693 ± 0.0747,0.4775 ± 0.0650
2,KNN,0.4700 ± 0.0780,0.4465 ± 0.0728,0.4380 ± 0.0805,0.4700 ± 0.0780
9,MLP,0.4275 ± 0.0595,0.4209 ± 0.0583,0.4272 ± 0.0608,0.4275 ± 0.0595
4,DecisionTree,0.4175 ± 0.0602,0.4187 ± 0.0598,0.4406 ± 0.0678,0.4175 ± 0.0602


TabDDPM - TSTR


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
7,GradientBoost,0.3875 ± 0.0860,0.3025 ± 0.0646,0.2871 ± 0.0546,0.3875 ± 0.0860
2,KNN,0.3775 ± 0.0759,0.3279 ± 0.0621,0.3123 ± 0.0566,0.3775 ± 0.0759
5,RandomForest,0.3300 ± 0.0780,0.2724 ± 0.0714,0.2841 ± 0.0569,0.3300 ± 0.0780
4,DecisionTree,0.3250 ± 0.0565,0.2703 ± 0.0596,0.2791 ± 0.0606,0.3250 ± 0.0565
6,ExtraTrees,0.3225 ± 0.0803,0.2645 ± 0.0810,0.2869 ± 0.0800,0.3225 ± 0.0803
3,NaiveBayes,0.2950 ± 0.0734,0.1754 ± 0.1057,0.1774 ± 0.1489,0.2950 ± 0.0734
9,MLP,0.2950 ± 0.0949,0.1412 ± 0.0868,0.0951 ± 0.0688,0.2950 ± 0.0949
8,AdaBoost,0.2600 ± 0.0412,0.1615 ± 0.0620,0.2554 ± 0.1394,0.2600 ± 0.0412
1,SVM-RBF,0.2500 ± 0.0000,0.1000 ± 0.0000,0.0625 ± 0.0000,0.2500 ± 0.0000
0,LogReg,0.2500 ± 0.0000,0.1000 ± 0.0000,0.0625 ± 0.0000,0.2500 ± 0.0000


TabDDPM - TRTR vs TSTR


...

,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
3,NaiveBayes,0.4525 ± 0.0399,0.3425 ± 0.0293,0.2820 ± 0.0306,0.4525 ± 0.0399
7,GradientBoost,0.4150 ± 0.0459,0.3703 ± 0.0391,0.3462 ± 0.0468,0.4150 ± 0.0459
4,DecisionTree,0.3875 ± 0.0626,0.3442 ± 0.0528,0.3274 ± 0.0455,0.3875 ± 0.0626
5,RandomForest,0.3875 ± 0.0460,0.3525 ± 0.0372,0.3376 ± 0.0310,0.3875 ± 0.0460
9,MLP,0.3750 ± 0.0527,0.3417 ± 0.0442,0.3347 ± 0.0426,0.3750 ± 0.0527
6,ExtraTrees,0.3700 ± 0.0563,0.3363 ± 0.0482,0.3248 ± 0.0379,0.3700 ± 0.0563
8,AdaBoost,0.3625 ± 0.0543,0.3314 ± 0.0491,0.3254 ± 0.0530,0.3625 ± 0.0543
2,KNN,0.3625 ± 0.0543,0.3237 ± 0.0472,0.3125 ± 0.0431,0.3625 ± 0.0543
0,LogReg,0.3525 ± 0.0617,0.3236 ± 0.0521,0.3151 ± 0.0503,0.3525 ± 0.0617
1,SVM-RBF,0.3475 ± 0.0583,0.3190 ± 0.0496,0.3133 ± 0.0477,0.3475 ± 0.0583


ForestDiffusion - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy ± SD_TRTR,Accuracy ± SD_TSTR
0,ForestDiffusion,GradientBoost,0.1100,0.123092,0.136810,0.1100,0.5250 ± 0.0527,0.4150 ± 0.0459
1,ForestDiffusion,SVM-RBF,0.1675,0.082103,0.031463,0.1675,0.5150 ± 0.0459,0.3475 ± 0.0583
2,ForestDiffusion,AdaBoost,0.1525,0.114536,0.095094,0.1525,0.5150 ± 0.0489,0.3625 ± 0.0543
3,ForestDiffusion,LogReg,0.1525,0.071120,0.017691,0.1525,0.5050 ± 0.0483,0.3525 ± 0.0617
4,ForestDiffusion,RandomForest,0.1000,0.111207,0.124785,0.1000,0.4875 ± 0.0637,0.3875 ± 0.0460
5,ForestDiffusion,NaiveBayes,0.0325,0.046000,0.092905,0.0325,0.4850 ± 0.0580,0.4525 ± 0.0399
6,ForestDiffusion,ExtraTrees,0.1075,0.132842,0.144462,0.1075,0.4775 ± 0.0650,0.3700 ± 0.0563
7,ForestDiffusion,KNN,0.1075,0.122791,0.125506,0.1075,0.4700 ± 0.0780,0.3625 ± 0.0543
8,ForestDiffusion,MLP,0.0525,0.079114,0.092515,0.0525,0.4275 ± 0.0595,0.3750 ± 0.0527
9,ForestDiffusion,DecisionTree,0.0300,0.074578,0.113206,0.0300,0.4175 ± 0.0602,0.3875 ± 0.0626


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,0.10125,0.095738,0.097444,0.10125
1,TabDDPM,0.17325,0.222678,0.209113,0.17325


In [10]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
